<a href="https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

*Unit of analysis: One row represents the daily performance of one content item for one client, identified by client_hash_id, content_hash_id, and report_date.*

*Time window: I use March 2026 as the development window. The decision-time features must come from information available before the outcome/decision moment. June 2026 is treated as the final/sealed month and is not used to develop the feature or label logic.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature:
* gsc_clicks
* gsc_impressions
* gsc_avg_position
* ga4_sessions
* content_visible_query_count

*Label: Future organic search clicks for the content item in the outcome window. This is the performance outcome I want to predict/rank using information that was available at the decision moment.*

Context:
* client_hash_id
* content_hash_id
* report_date
* content_type
* main_intent

*Excluded: future-period performance metrics used to construct the label. They are excluded from the feature set because they are not available at the decision moment and would cause target leakage.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
%pip -q install duckdb huggingface_hub

In [5]:
import os
import getpass
import duckdb

# Get HF token from environment or Colab Secret
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

# Create DuckDB connection
con = duckdb.connect()

# Give DuckDB access to Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Connection ready!")

Paste your Hugging Face READ token (hf_...): ··········
Connection ready!


In [6]:
REL = "hf://datasets/FlyRank/internship-warehouse"

print(REL)

hf://datasets/FlyRank/internship-warehouse


In [7]:
TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print(TABLES.keys())

dict_keys(['dim_clients', 'dim_content', 'fact_daily', 'fact_daily_sample', 'fact_query_90d'])


In [8]:
test = con.sql(f"""
    SELECT *
    FROM {TABLES["fact_daily"]}
    LIMIT 5
""").df()

test

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query1 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 5
"""

In [18]:
grain_check = con.sql(query1).df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


In [19]:
query2 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS client_count,
    COUNT(DISTINCT content_hash_id) AS content_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

window_check = con.sql(query2).df()

window_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,client_count,content_count,min_report_date,max_report_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [20]:
query3 = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS NULL
    ) AS gsc_missing_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS NULL
    ) AS ga4_missing_rows

FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

availability_check = con.sql(query3).df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,gsc_missing_rows,ga4_missing_rows
0,9841378,3611061,413966,0,3018741


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation — unbalanced history and source availability.**

The March 2026 slice does not represent an equally complete history for every
client or content item. Some rows may have limited historical data because
clients and content items enter the warehouse at different times. GSC and GA4
availability can also differ across rows, so missing values should not
automatically be interpreted as zero performance.

This means the features and comparisons are decision-support signals for the
observed data, not evidence of causal effects or complete historical
performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.